# Notebook 01 – Build Historical Dataset

Build a labelled feature dataset for All-Star prediction.

**What this notebook does:**
1. Fetches per-player game logs from the NBA API for each training season.
2. Aggregates stats from **Oct 1 through Feb 1** of each season (pre-All-Star cutoff).
3. Attaches All-Star labels from `allstar.csv`.
4. Saves the result to `data/historical_features.parquet`.

**Run once** – results are cached to disk, so subsequent runs are fast.

> **Prerequisites**: place `allstar.csv` in the `data/` folder (download from
> [Kaggle](https://www.kaggle.com/datasets/ethankeyes/nba-all-star-players-and-stats-1980-2022))
> before running this notebook.


In [1]:
import sys, subprocess


packages = [
    "numpy", "pandas", "nba_api", "unidecode",
    "scikit-learn", "imbalanced-learn", "xgboost",
    "matplotlib", "joblib", "pyarrow", "tqdm",
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Dependencies ready.")
                                                    

Dependencies ready.


In [2]:
import sys
from pathlib import Path

# Make sure src/ is on the path when running from notebooks/
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.data_fetcher import fetch_player_name



In [3]:
# ── Configuration ────────────────────────────────────────────────────────
# Seasons for which we build historical features.
SEASONS = [
    "2008-09", "2009-10", "2010-11", "2011-12",
    "2012-13", "2013-14", "2014-15", "2015-16",
    "2016-17", "2017-18", "2018-19", "2019-20",
    "2020-21", "2021-22",
]

DATA_DIR   = REPO_ROOT / "data"
CACHE_DIR  = DATA_DIR / "cache"
ALLSTAR_CSV = DATA_DIR / "allstar.csv"
OUTPUT_PATH = DATA_DIR / "historical_features.parquet"

CACHE_DIR.mkdir(parents=True, exist_ok=True)

## All-Star Dataset Preprocessing

The source CSV (`allstar.csv`) contains NBA All-Star data from **1980 through 2022** (2021-22 season).

**Preprocessing step:**
- Extract only the essential columns: `first`, `last`, `team`, `year`
- Remove all other statistical columns to keep the dataset clean
- Save the filtered dataset back to `allstar.csv`

This simplified dataset serves as the ground truth for labeling historical player seasons as All-Star or non-All-Star.

In [4]:
import pandas as pd

# Load and preprocess allstar.csv - keep only necessary columns
if ALLSTAR_CSV.exists():
    allstar_df = pd.read_csv(ALLSTAR_CSV)
    
    # Select only the required columns
    required_cols = ['first', 'last', 'team', 'year']
    
    # Check if all required columns exist
    missing_cols = [col for col in required_cols if col not in allstar_df.columns]
    if missing_cols:
        print(f"Warning: Missing columns in allstar.csv: {missing_cols}")
    else:
        # Keep only required columns and save back
        allstar_df = allstar_df[required_cols].copy()
        allstar_df.to_csv(ALLSTAR_CSV, index=False)
        print(f"allstar.csv preprocessed – {len(allstar_df)} rows with columns: {required_cols}")
        print(f"Year range: {allstar_df['year'].min()} to {allstar_df['year'].max()}")
else:
    print(f"ERROR: {ALLSTAR_CSV} not found!")
    print("Please download allstar.csv from:")
    print("  https://www.kaggle.com/datasets/ethankeyes/nba-all-star-players-and-stats-1980-2022")
    print("and place it in the data/ directory.")

allstar.csv preprocessed – 1003 rows with columns: ['first', 'last', 'team', 'year']
Year range: 1980 to 2022


<h2>collecting datas</h2>

To gather career statistics of players who played in the NBA, I used the NBA API.  
Link: [https://github.com/swar/nba_api?tab=readme-ov-file](https://github.com/swar/nba_api?tab=readme-ov-file)

Using this API, I obtained the `player_id` of all registered players and requested the career stats for each player to create a DataFrame. The career stats of each player consist of the following features:  
["PLAYER_ID", "SEASON_ID", "LEAGUE_ID", "TEAM_ID", "TEAM_ABBREVIATION", "PLAYER_AGE",  
"GP", "GS", "MIN", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT", "FTM", "FTA", "FT_PCT",  
"OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PTS"].  

These features will later be used to train an algorithm that predicts whether a player is selected as an All-Star. When using the NBA API, sending too many requests to the NBA website resulted in timeout errors. To address this, I created a function called `fetch_player_data` that retries requests with progressively longer `sleeptime` values up to a maximum of five attempts. This function allowed me to consolidate all players' career stats into a single DataFrame, which was then saved as a CSV file.  

For `player_id` 1630203 timeout errors persisted even after five attempts, so I manually added this player's career stats to the overall DataFrame.

In [ ]:
import pandas as pd
import time
from nba_api.stats.static import players
from nba_api.stats.endpoints import playercareerstats
import warnings
from 

warnings.filterwarnings('ignore', category=FutureWarning)


# get all players
players = players.get_players()

# get id of all players(5034 players)
ids = []
for player in players:
    ids.append(player["id"])

#create empty dataframe
columns = ["PLAYER_ID", "SEASON_ID", "LEAGUE_ID", "TEAM_ID", "TEAM_ABBREVIATION", "PLAYER_AGE",
           "GP", "GS", "MIN", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT", "FTM", "FTA", "FT_PCT",
           "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PTS"]
all_career = pd.DataFrame(columns=columns)

# get career of all players
for player_id in ids:
    data = fetch_player_data(player_id)
    if data is not None: 
        all_career = pd.concat([all_career, data], ignore_index=True)

print("all datas:", all_career.head())
all_career.to_csv("all_career.csv", index=False, encoding="utf-8")

print("successfully saved as CSV file: all_career.csv")


Error fetching data for player_id 76001: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error fetching data for player_id 76001: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error fetching data for player_id 76001: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error fetching data for player_id 76001: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error fetching data for player_id 76002: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error fetching data for player_id 76002: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error fetching data for player_id 76002: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error fetching data for player_id 76003: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed

## Team Name Unification

This block standardizes team abbreviations between the all_career dataset and allstar.csv to ensure accurate matching when labeling All-Star selections.

### Purpose

NBA team abbreviations have changed over time due to:
- Team relocations (e.g., Seattle SuperSonics to Oklahoma City Thunder)
- Franchise rebranding (e.g., New Jersey Nets to Brooklyn Nets)
- Inconsistent naming conventions in different data sources

Without unification, the same team-season record would fail to match between datasets, causing All-Star players to be incorrectly labeled as non-All-Stars.

### Process

**Step 1: Remove aggregate records**
- Filters out rows where team = "TOT" (total stats for players who played on multiple teams in one season)
- These aggregate records are not used for All-Star prediction

**Step 2: Identify mismatches**
- Compares unique team abbreviations in both datasets
- Reports teams that exist in only one dataset

**Step 3: Apply mappings**
- Uses two mapping dictionaries:
  - `known_mappings`: Converts all_career team names to standard abbreviations
  - `allstar_team_mapping`: Corrects inconsistencies in allstar.csv
- Only applies mappings for teams that actually appear in the mismatched lists

**Step 4: Verify results**
- Re-checks team lists after mapping
- Reports any remaining mismatches that require manual investigation


In [5]:

# Step 1: 이미 저장된 all_career.csv 불러오기
all_career = pd.read_csv(DATA_DIR / "all_career.csv")

# Step 2: 컬럼명 통일 (TEAM_ABBREVIATION → team, SEASON_ID → year)
all_career = all_career.rename(columns={
    "TEAM_ABBREVIATION": "team",
    "SEASON_ID": "year"
})

# Step 3: year를 정수로 변환 (예: "2019-20" → 2019)
if all_career["year"].dtype == 'object':
    all_career["year"] = all_career["year"].str[:4].astype(int)

print(f"Data loaded and normalized: {all_career.shape}")
print(f"Columns: {list(all_career.columns)}")



Data loaded and normalized: (30406, 27)
Columns: ['PLAYER_ID', 'year', 'LEAGUE_ID', 'TEAM_ID', 'team', 'PLAYER_AGE', 'GP', 'GS', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']


In [6]:
# Team name unification (requested version)

# 1) Remove TOT rows from all_career
all_career = all_career[all_career["team"] != "TOT"].copy()

# 2) Replace team codes in all_career (left -> right)
known_mappings = {
    "PHL": "PHI",   # Philadelphia
    "PHX": "PHO",   # Phoenix
    "GOS": "GSW",   # Golden State Warriors (old)
    "SAN": "SAS",   # San Antonio Spurs (old)
    "UTH": "UTA",   # Utah Jazz (old)
}
all_career["team"] = all_career["team"].replace(known_mappings)

# 3) Replace team codes in allstar.csv dataframe (left -> right)
allstar_team_mapping = {
    "BRK": "BKN",   # Brooklyn (typo)
    "CHO": "CHA",   # Charlotte
    "WSB": "WAS",   # Washington
}
allstar_df["team"] = allstar_df["team"].replace(allstar_team_mapping)

print("Done: TOT removed, all_career team names mapped, allstar_df team names mapped.")

Done: TOT removed, all_career team names mapped, allstar_df team names mapped.


## All-Star Label Assignment and Validation

This block adds All-Star labels to the career dataset and validates the accuracy of the labeling process.

### Process

**Step 1: Add player name columns**
- Creates empty `first` and `last` columns in the all_career dataset
- Fetches player names using the NBA API based on PLAYER_ID
- Converts names to standard English characters using unidecode

**Step 2: Column reordering**
- Moves player identification columns (PLAYER_ID, first, last) to the front
- Maintains logical column order for easier data inspection

**Step 3: All-Star label assignment**
- Initializes `allstar` column with default value 0 (non-All-Star)
- Creates a set of all-star tuples (first, last, year, team) for efficient lookup
- Matches each player-season record against the All-Star dataset
- Sets allstar = 1 for matching records

**Step 4: Save results**
- Exports the labeled dataset to `all_career_1980_with_allstar.csv`


## All-Star Label Assignment and Validation

This block adds All-Star labels to the career dataset and validates the accuracy of the labeling process.

### Process

**Step 1: Add player name columns**
- Creates empty `first` and `last` columns in the all_career dataset
- Fetches player names using the NBA API based on PLAYER_ID
- Converts names to standard English characters using unidecode

**Step 2: Column reordering**
- Moves player identification columns (PLAYER_ID, first, last) to the front
- Maintains logical column order for easier data inspection

**Step 3: All-Star label assignment**
- Initializes `allstar` column with default value 0 (non-All-Star)
- Creates a set of all-star tuples (first, last, year, team) for efficient lookup
- Matches each player-season record against the All-Star dataset
- Sets allstar = 1 for matching records

**Step 4: Save results**
- Exports the labeled dataset to `all_career_1980_with_allstar.csv`


In [7]:
import time
from nba_api.stats.static import players
from unidecode import unidecode


# Add 'first' and 'last' columns
all_career["first"] = None
all_career["last"] = None

# Fetch player names
print("\nFetching player names... (this may take a while)")
for index, row in all_career.iterrows():
    player_id = row["PLAYER_ID"]
    player_name = fetch_player_name(player_id)
    if player_name:
        all_career.at[index, "first"] = player_name["first_name"]
        all_career.at[index, "last"] = player_name["last_name"]

# Reorder columns
columns_order = ['PLAYER_ID', 'first', 'last'] + [col for col in all_career.columns if col not in ['PLAYER_ID', 'first', 'last']]
all_career = all_career[columns_order]

# Initialize allstar column with default value 0
all_career["allstar"] = 0

# Create a set of all-star tuples (first, last, year, team) for efficient lookup
allstar_set = set(zip(
    allstar_df["first"],
    allstar_df["last"],
    allstar_df["year"],
    allstar_df["team"]
))

# Check if each player record exists in the all-star dataset
# If match found, set allstar = 1, otherwise keep 0
all_career["allstar"] = all_career.apply(
    lambda row: 1 if (row["first"], row["last"], row["year"], row["team"]) in allstar_set else 0,
    axis=1
)

# Save final dataframe to data folder
output_path = DATA_DIR / "all_career_1980_with_allstar.csv"
all_career.to_csv(output_path, index=False, encoding="utf-8")




Fetching player names... (this may take a while)


## BLOCK 3: Name Standardization & Manual Exceptions

- Normalize player names between `allstar_df` and `all_career`.
- Apply manual fixes for known mismatches:
  - Magic Johnson (1991, LAL) remove
  - Jaren Jackson Jr (2022) rename
  - Yao Ming split name fix
  - Micheal Ray Richardson rename
  - Joe Barry Carroll rename
  - Steve → Steven Smith rename
- Rebuild labels with corrected `(first, last, year, team)` matching.
- Save:
  - `data/all_career_1980_with_allstar.csv`
  - `data/allstar_corrected.csv`

In [8]:
# ═══════════════════════════════════════════════════════════════════════
# BLOCK 3: Name Standardization + Manual Exceptions + Re-labeling
# (Merged version of old BLOCK 3-1 and 3-2)
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("BLOCK 3 - NAME STANDARDIZATION & MANUAL EXCEPTIONS (MERGED)")
print("="*70)

# Step 0: Create working copies
all_career_labeled = all_career.copy()
allstar_df = allstar_df.copy()

# ---------------------------------------------------------------------
# Step 1) Fix additional name format issues (from old BLOCK 3-1)
# ---------------------------------------------------------------------
print("\n[Step 1] Fixing additional name format issues")

# 1-1) Remove suffixes in all_career_labeled last names
print("\n1-1) Removing suffixes from all_career_labeled['last']")
suffixes = [" III", " Jr", " Sr", " II", " IV", " V"]
for suffix in suffixes:
    mask = all_career_labeled["last"].str.contains(suffix, na=False)
    if mask.any():
        cnt = int(mask.sum())
        all_career_labeled.loc[mask, "last"] = (
            all_career_labeled.loc[mask, "last"].str.replace(suffix, "", regex=False)
        )
        print(f"  ✓ Removed '{suffix}' from {cnt} row(s)")

# 1-2) Handle empty first names in allstar_df (e.g. ",Yao Ming")
print("\n1-2) Fixing empty first names in allstar_df")
empty_first = allstar_df["first"].isna() | (allstar_df["first"].astype(str).str.strip() == "")
if empty_first.any():
    cnt = int(empty_first.sum())
    allstar_df.loc[empty_first, "first"] = allstar_df.loc[empty_first, "last"]
    allstar_df.loc[empty_first, "last"] = ""
    print(f"  ✓ Moved last -> first for {cnt} row(s)")
else:
    print("  - No empty first names found")

# ---------------------------------------------------------------------
# Step 2) Manual exceptions (from old BLOCK 3-2 + your new 3 cases)
# ---------------------------------------------------------------------
print("\n[Step 2] Applying manual exceptions to allstar_df / all_career_labeled")

# 2-0) Yao Ming format in all_career_labeled: ("", "Yao Ming") -> ("Yao", "Ming")
yao_ming_original_mask = (
    ((all_career_labeled["first"] == "") | (all_career_labeled["first"].isna())) &
    (all_career_labeled["last"].str.contains("Yao Ming", na=False, case=False))
)
if yao_ming_original_mask.any():
    cnt = int(yao_ming_original_mask.sum())
    all_career_labeled.loc[yao_ming_original_mask, "first"] = "Yao"
    all_career_labeled.loc[yao_ming_original_mask, "last"] = "Ming"
    print(f"  ✓ Yao in all_career_labeled fixed: {cnt} row(s)")

# 2-1) Remove Magic Johnson 1991 LAL
magic_johnson_mask = (
    allstar_df["first"].str.contains("Magic|Earvin", na=False, case=False) &
    allstar_df["last"].str.contains("Johnson", na=False, case=False) &
    (allstar_df["year"] == 1991) &
    (allstar_df["team"] == "LAL")
)
if magic_johnson_mask.any():
    cnt = int(magic_johnson_mask.sum())
    allstar_df = allstar_df.loc[~magic_johnson_mask].reset_index(drop=True)
    print(f"  ✓ Removed Magic Johnson (1991, LAL): {cnt} row(s)")

# 2-2) Jaren Jackson Jr -> Jaren / Jackson.
jaren_jackson_mask = (
    allstar_df["first"].str.contains("Jaren", na=False, case=False) &
    allstar_df["last"].str.contains("Jackson", na=False, case=False) &
    (allstar_df["year"] == 2022)
)
if jaren_jackson_mask.any():
    cnt = int(jaren_jackson_mask.sum())
    allstar_df.loc[jaren_jackson_mask, "first"] = "Jaren"
    allstar_df.loc[jaren_jackson_mask, "last"] = "Jackson."
    print(f"  ✓ Jaren Jackson Jr fixed: {cnt} row(s)")

# 2-3) Yao Ming in allstar_df -> Yao / Ming
yao_ming_mask_allstar = allstar_df["last"].str.contains("Yao Ming", na=False, case=False)
if yao_ming_mask_allstar.any():
    cnt = int(yao_ming_mask_allstar.sum())
    allstar_df.loc[yao_ming_mask_allstar, "first"] = "Yao"
    allstar_df.loc[yao_ming_mask_allstar, "last"] = "Ming"
    print(f"  ✓ Yao Ming in allstar_df fixed: {cnt} row(s)")

# 2-4) NEW: Micheal Ray Richardson (allstar -> all_career format)
mrr_mask = (
    allstar_df["first"].str.contains("Micheal", na=False, case=False) &
    allstar_df["last"].str.contains("Ray Richardson", na=False, case=False) &
    allstar_df["year"].isin([1980, 1981, 1984]) &
    allstar_df["team"].isin(["NYK", "NJN"])
)
if mrr_mask.any():
    cnt = int(mrr_mask.sum())
    allstar_df.loc[mrr_mask, "first"] = "Micheal Ray"
    allstar_df.loc[mrr_mask, "last"] = "Richardson"
    print(f"  ✓ Micheal Ray Richardson fixed: {cnt} row(s)")

# 2-5) NEW: Joe Barry Carroll (allstar -> all_career format)
jbc_mask = (
    allstar_df["first"].str.contains("Joe", na=False, case=False) &
    allstar_df["last"].str.contains(r"Barry,?\s*Carroll", na=False, case=False, regex=True) &
    (allstar_df["year"] == 1986) &
    (allstar_df["team"] == "GSW")
)
if jbc_mask.any():
    cnt = int(jbc_mask.sum())
    allstar_df.loc[jbc_mask, "first"] = "Joe Barry"
    allstar_df.loc[jbc_mask, "last"] = "Carroll"
    print(f"  ✓ Joe Barry Carroll fixed: {cnt} row(s)")

# 2-6) NEW: Steve Smith -> Steven Smith (allstar -> all_career format)
steven_smith_mask = (
    allstar_df["first"].str.contains("^Steve$", na=False, case=False, regex=True) &
    allstar_df["last"].str.contains("^Smith$", na=False, case=False, regex=True) &
    (allstar_df["year"] == 1997) &
    (allstar_df["team"] == "ATL")
)
if steven_smith_mask.any():
    cnt = int(steven_smith_mask.sum())
    allstar_df.loc[steven_smith_mask, "first"] = "Steven"
    allstar_df.loc[steven_smith_mask, "last"] = "Smith"
    print(f"  ✓ Steve -> Steven Smith fixed: {cnt} row(s)")

# ---------------------------------------------------------------------
# Step 3) Find unmatched allstars after fixes
# ---------------------------------------------------------------------
print("\n[Step 3] Checking unmatched All-Stars after fixes")
unmatched_allstars = []
for _, row in allstar_df.iterrows():
    match_found = all_career_labeled[
        (all_career_labeled["first"] == row["first"]) &
        (all_career_labeled["last"] == row["last"]) &
        (all_career_labeled["year"] == row["year"]) &
        (all_career_labeled["team"] == row["team"])
    ]
    if len(match_found) == 0:
        unmatched_allstars.append(row)

print(f"  Remaining unmatched All-Stars: {len(unmatched_allstars)}")
if unmatched_allstars:
    print("  Unmatched list:")
    for i, r in enumerate(unmatched_allstars, 1):
        f = r["first"] if pd.notna(r["first"]) else ""
        l = r["last"] if pd.notna(r["last"]) else ""
        print(f"    {i}. '{f}' '{l}' ({r['year']}, {r['team']})")

# ---------------------------------------------------------------------
# Step 4) Re-label allstar on all_career_labeled
# ---------------------------------------------------------------------
print("\n[Step 4] Re-labeling all_career_labeled['allstar']")
allstar_set = set(zip(
    allstar_df["first"],
    allstar_df["last"],
    allstar_df["year"],
    allstar_df["team"]
))

all_career_labeled["allstar"] = all_career_labeled.apply(
    lambda row: 1 if (row["first"], row["last"], row["year"], row["team"]) in allstar_set else 0,
    axis=1
)

# ---------------------------------------------------------------------
# Step 5) Validation + save
# ---------------------------------------------------------------------
print("\n[Step 5] Validation & save")
allstar_count_final = int((all_career_labeled["allstar"] == 1).sum())
allstar_df_rows = int(len(allstar_df))

print(f"  Expected All-Star count: {allstar_df_rows}")
print(f"  Actual All-Star count:   {allstar_count_final}")
print(f"  Match rate: {allstar_count_final}/{allstar_df_rows} ({100*allstar_count_final/allstar_df_rows:.1f}%)")

if allstar_count_final == allstar_df_rows:
    print("  ✓ Perfect match! All All-Star records labeled correctly.")
else:
    print(f"  ⚠ Still {allstar_df_rows - allstar_count_final} unmatched")

# Optional: quick checks for your 3 new manual cases
print("\nQuick verification for new manual cases:")
checks = [
    ("Micheal Ray", "Richardson", [1980, 1981, 1984], ["NYK", "NJN"]),
    ("Joe Barry", "Carroll", [1986], ["GSW"]),
    ("Steven", "Smith", [1997], ["ATL"]),
]
for first_name, last_name, years, teams in checks:
    sub = all_career_labeled[
        (all_career_labeled["first"] == first_name) &
        (all_career_labeled["last"] == last_name) &
        (all_career_labeled["year"].isin(years)) &
        (all_career_labeled["team"].isin(teams))
    ][["first", "last", "year", "team", "allstar"]].sort_values(["year", "team"])
    print(f"\n  {first_name} {last_name}:")
    if len(sub) == 0:
        print("    (no rows found)")
    else:
        print(sub.to_string(index=False))

# Save outputs
output_path = DATA_DIR / "all_career_1980_with_allstar.csv"
all_career_labeled.to_csv(output_path, index=False, encoding="utf-8")

allstar_corrected_path = DATA_DIR / "allstar_corrected.csv"
allstar_df.to_csv(allstar_corrected_path, index=False, encoding="utf-8")

print("\nSaved:")
print(f"  - {output_path}")
print(f"  - {allstar_corrected_path}")
print("="*70)


BLOCK 3 - NAME STANDARDIZATION & MANUAL EXCEPTIONS (MERGED)

[Step 1] Fixing additional name format issues

1-1) Removing suffixes from all_career_labeled['last']
  ✓ Removed ' III' from 96 row(s)
  ✓ Removed ' Jr' from 222 row(s)
  ✓ Removed ' Sr' from 16 row(s)
  ✓ Removed ' II' from 30 row(s)
  ✓ Removed ' IV' from 13 row(s)
  ✓ Removed ' V' from 1 row(s)

1-2) Fixing empty first names in allstar_df
  - No empty first names found

[Step 2] Applying manual exceptions to allstar_df / all_career_labeled
  ✓ Yao in all_career_labeled fixed: 8 row(s)
  ✓ Removed Magic Johnson (1991, LAL): 1 row(s)
  ✓ Jaren Jackson Jr fixed: 1 row(s)
  ✓ Micheal Ray Richardson fixed: 3 row(s)
  ✓ Joe Barry Carroll fixed: 1 row(s)
  ✓ Steve -> Steven Smith fixed: 1 row(s)

[Step 3] Checking unmatched All-Stars after fixes
  Remaining unmatched All-Stars: 0

[Step 4] Re-labeling all_career_labeled['allstar']

[Step 5] Validation & save
  Expected All-Star count: 1002
  Actual All-Star count:   1002
  Matc

In [9]:
import pandas as pd

# Load both files
allstar_corrected = pd.read_csv(DATA_DIR / "allstar_corrected.csv")
allstar_original = pd.read_csv(DATA_DIR / "allstar.csv")

# Count allstar=1 in all_career_labeled (saved as allstar_corrected)
allstar_corrected_count = len(allstar_corrected)

# Count rows in original allstar.csv
allstar_original_count = len(allstar_original)

# Display comparison
print("="*70)
print("ALLSTAR DATASET COMPARISON")
print("="*70)
print(f"\nallstar_corrected.csv (expected All-Star count): {allstar_corrected_count} rows")
print(f"allstar.csv (original All-Star count):          {allstar_original_count} rows")
print(f"\nDifference: {allstar_corrected_count - allstar_original_count} rows")

if allstar_corrected_count == allstar_original_count:
    print("\n✓ Perfect match! All records accounted for.")
else:
    print(f"\n⚠ Mismatch: {abs(allstar_corrected_count - allstar_original_count)} rows difference")

print("="*70)

ALLSTAR DATASET COMPARISON

allstar_corrected.csv (expected All-Star count): 1002 rows
allstar.csv (original All-Star count):          1003 rows

Difference: -1 rows

⚠ Mismatch: 1 rows difference


In [10]:
import pandas as pd

# Load both files
all_career_labeled = pd.read_csv(DATA_DIR / "all_career_1980_with_allstar.csv")
allstar_original = pd.read_csv(DATA_DIR / "allstar.csv")

# Find rows in allstar_original that are NOT in all_career_labeled (allstar=1)
print("="*70)
print("MISSING ALL-STAR RECORDS")
print("="*70)

missing_count = 0
for _, row in allstar_original.iterrows():
    first = row["first"]
    last = row["last"]
    year = row["year"]
    team = row["team"]
    
    # Check if this all-star exists in all_career_labeled with allstar=1
    match = all_career_labeled[
        (all_career_labeled["first"] == first) &
        (all_career_labeled["last"] == last) &
        (all_career_labeled["year"] == year) &
        (all_career_labeled["team"] == team) &
        (all_career_labeled["allstar"] == 1)
    ]
    
    if len(match) == 0:
        missing_count += 1
        print(f"\n[Missing {missing_count}] '{first}' '{last}' ({year}, {team})")
        
        # Check if player exists but not labeled as all-star
        player_exists = all_career_labeled[
            (all_career_labeled["first"] == first) &
            (all_career_labeled["last"] == last) &
            (all_career_labeled["year"] == year) &
            (all_career_labeled["team"] == team)
        ]
        if len(player_exists) > 0:
            print(f"  → Player found in all_career but allstar={int(player_exists['allstar'].iloc[0])}")
        else:
            print(f"  → Player NOT found in all_career dataset at all")

print(f"\n\nTotal missing All-Stars: {missing_count}")
print("="*70)

MISSING ALL-STAR RECORDS

[Missing 1] 'Micheal' 'Ray Richardson' (1980, NYK)
  → Player NOT found in all_career dataset at all

[Missing 2] 'Micheal' 'Ray Richardson' (1981, NYK)
  → Player NOT found in all_career dataset at all

[Missing 3] 'Jeff' 'Ruland' (1983, WSB)
  → Player NOT found in all_career dataset at all

[Missing 4] 'Micheal' 'Ray Richardson' (1984, NJN)
  → Player NOT found in all_career dataset at all

[Missing 5] 'Jeff' 'Malone' (1985, WSB)
  → Player NOT found in all_career dataset at all

[Missing 6] 'Joe' 'Barry,Carroll' (1986, GSW)
  → Player NOT found in all_career dataset at all

[Missing 7] 'Moses' 'Malone' (1986, WSB)
  → Player NOT found in all_career dataset at all

[Missing 8] 'Jeff' 'Malone' (1986, WSB)
  → Player NOT found in all_career dataset at all

[Missing 9] 'Moses' 'Malone' (1987, WSB)
  → Player NOT found in all_career dataset at all

[Missing 10] 'Bernard' 'King' (1990, WSB)
  → Player NOT found in all_career dataset at all

[Missing 11] 'Magic' 